# Aula 04 - Padrão de Design de Uso de Ferramentas

Nesta aula, irá aprender o padrão de design **Uso de Ferramentas** para agentes de IA utilizando o Microsoft Agent Framework (Python). Cobrimos:

- Definir ferramentas como funções com o decorador `@tool` e parâmetros tipados
- Fornecer esquemas das ferramentas para que o modelo saiba o que cada ferramenta faz
- Controlar a execução das ferramentas com `approval_mode`
- Devolver **output estruturado** através de modelos Pydantic e `response_format`

O cenário é um **agente de reservas de viagem** que pode pesquisar destinos, verificar disponibilidade e obter informações de voos.


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Definir Ferramentas com o Decorador @tool

O decorador `@tool` transforma uma função Python simples numa ferramenta que um agente pode chamar.
Pontos chave:

- A **docstring** torna-se na descrição da ferramenta que o modelo vê.
- As **anotações de tipo** (incluindo `Annotated` com descrições) definem o esquema da ferramenta.
- `approval_mode` controla se o utilizador deve aprovar cada chamada antes de esta ser executada.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Criar um Agente com Múltiplas Ferramentas

Passe as três ferramentas ao cliente para que o modelo possa invocar quaisquer que precisar para responder à pergunta do utilizador.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Saída Estruturada com Ferramentas

Ao passar um modelo Pydantic como `response_format` nas opções de execução, o agente é obrigado a devolver um objeto JSON com um tipo bem definido em vez de texto livre. Isto é útil quando o código subsequente precisa de consumir o resultado programaticamente. O modelo analisado está disponível no atributo `value` da resposta:


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available.",
    options={"response_format": TravelPlan},
)
if response:
    plan = response.value
    for rec in plan.recommendations:
        status = "available" if rec.available else "unavailable"
        print(
            f"{rec.destination}: {status} - {rec.flight_details} "
            f"(~${rec.estimated_cost})"
        )

## Padrões de Aprovação de Ferramentas

O parâmetro `approval_mode` em `@tool` controla se as chamadas de ferramentas requerem aprovação humana antes de serem executadas:

| Modo | Comportamento |
|---|---|
| `"never_require"` | A ferramenta executa automaticamente — não é necessária confirmação do utilizador. |
| `"always_require"` | Cada chamada deve ser aprovada pelo utilizador antes de ser executada. |

Use `"always_require"` para ferramentas que tenham efeitos secundários (ex. reservar um voo, cobrar um cartão de crédito) para que um humano esteja sempre envolvido.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Resumo

Nesta aula aprendeu a:

1. **Definir ferramentas** usando o decorador `@tool` com parâmetros tipados e docstrings que servem como o esquema da ferramenta.
2. **Compor múltiplas ferramentas** para que o agente possa chamá-las em sequência para responder a consultas complexas.
3. **Devolver output estruturado** ao passar um modelo Pydantic como `response_format`.
4. **Controlar a aprovação da ferramenta** com `approval_mode` para manter um humano no circuito para operações sensíveis.

Estes padrões formam a base para construir agentes fiáveis, prontos para produção, que podem interagir com sistemas externos de forma segura.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
